# setup -> clone the repo, install without touching torch, confirm the gpus, print the versions, run the public tests

## imports

In [ ]:
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError
import json
import platform
import shutil
import subprocess
import sys

OUTPUT_DIR = Path('./outputs/setup')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## global env variables ->

In [ ]:
#the defaults are the manual's kaggle setup, so on a kaggle t4 x2 notebook this runs unchanged. the modal runner
#overrides only the expected gpus, every reported run there used one rtx pro 6000
REPO_URL = 'https://github.com/mujeeb-asad/AGAIA-PA2.git'
CLONE_DIR = './AGAIA-PA2' #relative to the working directory, /kaggle/working on kaggle
EXPECTED_GPUS = 2
EXPECTED_GPU_NAME = 'T4' #matched as a substring of the device name

#the manual's install line, word for word. torch is deliberately not in it, the platform's own cuda build is kept
PACKAGES = ['transformers>=4.43.0,<5', 'datasets>=2.18,<5', 'huggingface_hub', 'accelerate', 'tqdm', 'pyyaml',
            'sentencepiece', 'safetensors', 'pytest']

In [ ]:
#runs a shell command and shows its output in the notebook. check=False is for the one step that may fail harmlessly
def run(command, cwd=None, check=True):
    print(f'$ {" ".join(command)}')
    result = subprocess.run(command, cwd=cwd, capture_output=True, text=True)
    print((result.stdout + result.stderr).strip())
    if check and result.returncode != 0:
        raise RuntimeError(f'command failed with exit code {result.returncode}')
    return result

#the version an installed package reports, without importing it. none when it is not installed
def installed(package):
    try:
        return version(package)
    except PackageNotFoundError:
        return None

## 1. clone the repository

In [ ]:
clone_dir = Path(CLONE_DIR).resolve()
if clone_dir.exists():
    shutil.rmtree(clone_dir) #a fresh clone every time, so this checks the pushed repo and not a stale local copy
run(['git', 'clone', '--depth', '1', REPO_URL, str(clone_dir)])
commit = run(['git', 'rev-parse', 'HEAD'], cwd=clone_dir).stdout.strip()
print(f'cloned {REPO_URL} at {commit}')

## 2. install the dependencies without reinstalling pytorch

In [ ]:
torch_before = installed('torch')
print(f'preinstalled torch {torch_before}')

#an old torchao can conflict with peft when lora adapters are loaded. absent on most images, so failure is fine here
run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchao'], check=False)
run([sys.executable, '-m', 'pip', 'install', '-q', *PACKAGES])
#--no-deps -> the repo becomes importable without pip being allowed to replace anything already installed
run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(clone_dir), '--no-deps'])

torch_after = installed('torch')
assert torch_after == torch_before, f'torch changed from {torch_before} to {torch_after}, the install line pulled it in'
print(f'torch untouched: {torch_after}')

#a fresh interpreter in another directory, so the import can only succeed through the editable install. the
#package is alignment, submission_adapter.py is a root level file that the tests import with the repo root on the path
run([sys.executable, '-c', 'import alignment; print("alignment from", alignment.__file__)'], cwd='/')

## 3. confirm the gpus

In [ ]:
run(['nvidia-smi'])

In [ ]:
#imported only now, after the install, so the versions printed below are the ones every later notebook gets
import torch
import transformers
import datasets

assert torch.cuda.is_available(), 'no cuda device visible, switch the accelerator on'
gpus = []
for index in range(torch.cuda.device_count()):
    properties = torch.cuda.get_device_properties(index)
    gpus.append({'index': index, 'name': properties.name, 'memory_gib': round(properties.total_memory / 2**30, 1),
                 'compute_capability': f'{properties.major}.{properties.minor}'})
    print(f'cuda:{index}  {properties.name}  {gpus[-1]["memory_gib"]} GiB  sm {gpus[-1]["compute_capability"]}')

assert len(gpus) == EXPECTED_GPUS, f'expected {EXPECTED_GPUS} gpus, found {len(gpus)}'
assert all(EXPECTED_GPU_NAME in gpu['name'] for gpu in gpus), f'expected {EXPECTED_GPU_NAME} gpus'
print(f'confirmed {EXPECTED_GPUS} x {EXPECTED_GPU_NAME}')

#a tiny matmul on every device, visible is not the same as usable
for gpu in gpus:
    x = torch.randn(1024, 1024, device=f'cuda:{gpu["index"]}')
    torch.cuda.synchronize(gpu['index'])
    print(f'cuda:{gpu["index"]} matmul ok, checksum {(x @ x).sum().item():.1f}')

## 4. package versions

In [ ]:
environment = {
    'repo': REPO_URL, 'commit': commit,
    'python': platform.python_version(),
    'torch': torch.__version__, 'cuda': torch.version.cuda, 'cudnn': torch.backends.cudnn.version(),
    'transformers': transformers.__version__, 'datasets': datasets.__version__,
    #not imported above, read from the install metadata. trl and peft are only present where the persona task runs
    **{package: installed(package) for package in ['accelerate', 'huggingface_hub', 'safetensors', 'trl', 'peft']},
    'gpus': gpus,
}
for key, value in environment.items():
    if key != 'gpus':
        print(f'{key:16s} {value}')

## 5. public adapter tests

In [ ]:
#the manual's own command, from the clone's root. the whole tests folder also covers the edge case tests added here
run([sys.executable, '-m', 'pytest', '-q', 'tests/'], cwd=clone_dir)

In [ ]:
(OUTPUT_DIR / 'environment.json').write_text(json.dumps(environment, indent=2))
print(f'wrote {OUTPUT_DIR / "environment.json"}')